# 05 · Creating Composition Arcs

> Companion notebook to `docs/creating-composition-arcs/` from the Learn OpenUSD curriculum.

This is the largest module in the course. It walks through every composition arc OpenUSD offers and how they combine via LIVRPS strength ordering. Use the notebook's table of contents (the outline panel in JupyterLab / VS Code) to jump between sections — you do not have to run it top-to-bottom in a single sitting.

## Learning objectives

By the end of this notebook you will be able to:

- Implement **layers** and **sublayers** to organize USD scene data across workstreams.
- Create and manage **references** and **payloads** for modular, deferrable asset composition.
- Apply proper **encapsulation** techniques so referenced assets survive the path-translation boundary.
- Design and author **variant sets** for flexible asset variations.
- Use **inherits** and **specializes** arcs for broadcast overrides and fallback values.
- Apply **LIVRPS strength ordering** rules to predict and control composition results.
- Debug composition using prim / layer stacks and Python introspection.

## How to use this notebook

Every cell writes into `./_assets/` next to the notebook — safe to delete at any time. Wherever the source exercise uses pre-authored files on disk, this notebook copies them from `docs/exercise_content/composition_arcs/` into `_assets/composition_arcs/` so the lessons are self-contained. If those source files are missing, the asset-dependent cells will print a warning and skip gracefully instead of crashing.

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9


pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets


In [2]:
# Try the repo's helper; fall back to an inline version if running outside the uv env.
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


In [3]:
# Copy the shared exercise content into _assets/ so every section below is self-contained.
# The source lives in docs/exercise_content/composition_arcs/. We walk upward from the
# notebook directory (and its parents) to find the repo root, then copy whatever exists.
# If the source is missing (e.g. someone downloaded only the notebook), we print a warning
# instead of crashing. Individual cells additionally check for the specific sub-exercise
# files they need, so the notebook runs clean even when only part of the tree is present.
import shutil
from pathlib import Path

ASSET_ROOT = NB_DIR / "_assets" / "composition_arcs"

def _find_comp_arcs():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in [base / "docs" / "exercise_content" / "composition_arcs",
                          base.parent / "docs" / "exercise_content" / "composition_arcs"]:
            if candidate.exists():
                return candidate
    return None

EXERCISE_SRC = _find_comp_arcs()

if EXERCISE_SRC is None:
    print("WARNING: could not find docs/exercise_content/composition_arcs/.")
    print("         Cells that depend on shipped exercise files will skip themselves.")
    HAVE_EXERCISE_CONTENT = False
else:
    if not ASSET_ROOT.exists():
        shutil.copytree(EXERCISE_SRC, ASSET_ROOT)
        print(f"Copied {EXERCISE_SRC}\n    ->  {ASSET_ROOT}")
    else:
        print(f"Already present: {ASSET_ROOT}")
    HAVE_EXERCISE_CONTENT = True

# Friendlier alias used by downstream cells.
HAVE_COMP_ARCS = HAVE_EXERCISE_CONTENT

def asset_exists(rel_path: str) -> bool:
    """Return True if a specific sub-path exists under the staged _assets/composition_arcs/."""
    return (ASSET_ROOT / rel_path).exists()

def require_assets(*rel_paths: str) -> bool:
    """Guard for cells that need shipped exercise files.

    Called with no arguments it only checks that the top-level exercise tree was
    staged. Called with one or more relative paths (relative to ASSET_ROOT) it
    additionally verifies each path exists; if any are missing it prints a skip
    message and returns False so the caller can short-circuit cleanly.
    """
    if not HAVE_COMP_ARCS:
        print("Skipping: shipped exercise content not available.")
        return False
    missing = [rp for rp in rel_paths if not asset_exists(rp)]
    if missing:
        for rp in missing:
            print(f"Skipping: asset not present in repo -> {rp}")
        return False
    return True

Copied /Users/stranzersweb/Projects/LearnOpenUSD/docs/exercise_content/composition_arcs
    ->  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs


## 5.0 Prim composition recap

A **prim** is the primary container object in USD. Each prim is assembled from one or more **prim specs** (plus **property specs** for its attributes and relationships) that live on **layers**. Every spec carries **opinions** — authored values that the composition engine merges into a single resolved prim on the stage.

You interact with specs through the `Sdf` (Scene Description Foundations) API: `Sdf.Layer`, `Sdf.PrimSpec`, `Sdf.AttributeSpec`, etc. Composition is the process that turns a pile of specs across many layers into one composed prim with one winning value per property. The rest of this notebook is about the arcs (sublayers, references, payloads, variants, inherits, specializes) that bring those specs together and the rules (LIVRPS) that decide who wins.

In [4]:
from pxr import Usd, UsdGeom, Sdf
from pathlib import Path

demo_path = NB_DIR / "_assets" / "05_prim_composition_demo.usda"
stage = Usd.Stage.CreateNew(str(demo_path))
sphere = UsdGeom.Sphere.Define(stage, "/World/Sphere")
sphere.GetRadiusAttr().Set(2.0)
stage.GetRootLayer().Save()

# Poke at the specs directly through the Sdf API so the distinction between
# "spec" (per-layer authored data) and "composed prim" (what you see on the stage)
# is concrete.
root_layer = stage.GetRootLayer()
prim_spec = root_layer.GetPrimAtPath("/World/Sphere")
print("Prim spec type     :", type(prim_spec).__name__)
print("Prim spec specifier:", prim_spec.specifier)
print("Property specs     :", [p.name for p in prim_spec.properties])
print("Composed radius    :", sphere.GetRadiusAttr().Get())

Prim spec type     : PrimSpec
Prim spec specifier: Sdf.SpecifierDef
Property specs     : ['radius']
Composed radius    : 2.0


**What just happened:** we authored a single sphere with `radius = 2.0`. `Sdf.PrimSpec` shows the raw opinion on the root layer; `UsdGeom.Sphere.GetRadiusAttr().Get()` returns the value *after* composition. With only one layer in play, the two are identical — the sections below introduce arcs that make them diverge.

## 5.1 Sublayers

**Sublayers** are an ordered list of layers that stack on top of each other like transparencies: the first entry is the strongest, each subsequent entry is weaker, and when multiple layers speak about the same prim the stronger layer wins. All contents of a sublayer are *included* into the destination layer with no path remapping — the namespaces are merged verbatim.

Use sublayers when multiple workstreams (lighting, layout, shading, FX) want to contribute to the same scene without stepping on each other. Each workstream owns one layer; a thin root layer sublayers them together.

![sublayers diagram](../docs/images/composition-arcs/image64.png)

### 5.1.1 Simple sublayer example

The shipped `sublayers/simple_example/` folder contains a root layer (`sublayers_simple.usd`) that sublayers `sublayerB.usd` over `sublayerA.usd`. `sublayerA` defines a `Cube` of size 100; `sublayerB` defines a `Sphere` and an `over` that resizes the cube to 50. Because `sublayerB` is listed first, its cube-size opinion wins.

In [5]:
from pxr import Usd, Sdf
from pathlib import Path

if require_assets("sublayers/simple_example/sublayers_simple.usd"):
    simple_root = ASSET_ROOT / "sublayers" / "simple_example" / "sublayers_simple.usd"
    stage = Usd.Stage.Open(str(simple_root))
    root_layer = stage.GetRootLayer()
    print("Root layer subLayerPaths (strongest first):")
    for p in root_layer.subLayerPaths:
        print("  ", p)

    print("\nPrims on the composed stage:")
    for prim in stage.Traverse():
        print("  ", prim.GetPath(), "(", prim.GetTypeName() or "<over>", ")")

    cube = stage.GetPrimAtPath("/World/Geometry/Cube")
    if cube and cube.IsValid():
        size_attr = cube.GetAttribute("size")
        if size_attr:
            print("\nComposed Cube.size   :", size_attr.Get())
            print("Property stack (opinions strongest -> weakest):")
            for spec in size_attr.GetPropertyStack(Usd.TimeCode.Default()):
                print("  ", spec.layer.identifier, "->", spec.default)

Root layer subLayerPaths (strongest first):
   ./sublayerB.usd
   ./sublayerA.usd

Prims on the composed stage:
   /World ( Xform )
   /World/Geometry ( Scope )
   /World/Geometry/Cube ( Cube )
   /World/Geometry/Sphere ( Sphere )

Composed Cube.size   : 100.0
Property stack (opinions strongest -> weakest):
   /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/sublayers/simple_example/sublayerA.usd -> 100.0


**What just happened:** the composed cube ends up with the value from `sublayerB.usd` because it is sublayered *before* (i.e. stronger than) `sublayerA.usd`. `GetPropertyStack()` is the single best debugging tool for composition questions — it returns every authored opinion in strength order and tells you exactly which layer a value came from.

### 5.1.2 Authoring sublayers with the Sdf API

Authoring sublayers is just list manipulation on `Sdf.Layer.subLayerPaths`. The exercise script builds a `my_skyscraper.usda` that composes a shading layer over a geometry layer:

In [6]:
from pxr import Usd, Sdf, UsdGeom
from pathlib import Path

if require_assets("sublayers/exercise/contents/geometry.usd",
                  "sublayers/exercise/contents/shading.usd"):
    working_dir = ASSET_ROOT / "sublayers" / "exercise"
    out_path = working_dir / "my_skyscraper.usda"
    # Make the script idempotent across repeated notebook runs.
    if out_path.exists():
        out_path.unlink()

    stage = Usd.Stage.CreateNew(str(out_path))
    UsdGeom.SetStageMetersPerUnit(stage, UsdGeom.LinearUnits.centimeters)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)

    root_layer: Sdf.Layer = stage.GetRootLayer()
    # First appended sublayer is the STRONGEST.
    root_layer.subLayerPaths.append("./contents/shading.usd")
    root_layer.subLayerPaths.append("./contents/geometry.usd")
    stage.Save()

    print("Wrote:", out_path)
    print("\n--- my_skyscraper.usda ---")
    print(out_path.read_text())

    composed = Usd.Stage.Open(str(out_path))
    print("\nComposed prims (first 12):")
    for i, prim in enumerate(composed.Traverse()):
        if i >= 12:
            print("  ...")
            break
        print("  ", prim.GetPath(), prim.GetTypeName())

Wrote: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/sublayers/exercise/my_skyscraper.usda

--- my_skyscraper.usda ---
#usda 1.0
(
    metersPerUnit = 0.01
    subLayers = [
        @./contents/shading.usd@,
        @./contents/geometry.usd@
    ]
    upAxis = "Y"
)



Composed prims (first 12):
   /World Xform
   /World/Geometry Scope
   /World/Geometry/sign_billboard Mesh
   /World/Geometry/sign_billboard/border GeomSubset
   /World/Geometry/sign_billboard/roof GeomSubset
   /World/Geometry/sign_billboard/subset_defaultMat GeomSubset
   /World/Geometry/skyscraperF Mesh
   /World/Geometry/skyscraperF/border GeomSubset
   /World/Geometry/skyscraperF/window GeomSubset
   /World/Geometry/skyscraperF/subset_defaultMat GeomSubset
   /World/Geometry/skyscraperF/door GeomSubset
   /World/Looks Scope
  ...


**What just happened:** two independent workstream layers (geometry and shading) were fused into one stage with a two-line API call. Either workstream can keep editing their layer; the root composition updates on reload without either team merging anything manually.

### 5.1.3 Sublayer FAQ

A few common questions:

- **What is `subLayerPaths`?** It is the Python-side property on `Sdf.Layer` that represents the sublayer list. You edit it with normal list operations (`append`, `insert`, `remove`).
- **How do I remove a sublayer?** `root_layer.subLayerPaths.remove("./contents/shading.usd")`.
- **How do I walk nested sublayers?** Grab `layer.subLayerPaths` recursively — sublayers can themselves have sublayers, and the entire recursive set forms one *layer stack*.

In [7]:
from pxr import Sdf

def walk_sublayers(layer: "Sdf.Layer", depth: int = 0):
    print("  " * depth + layer.identifier)
    for rel in layer.subLayerPaths:
        resolved = Sdf.Layer.FindOrOpenRelativeToLayer(layer, rel)
        if resolved:
            walk_sublayers(resolved, depth + 1)
        else:
            print("  " * (depth + 1) + f"<unresolved: {rel}>")

_my_sky = ASSET_ROOT / "sublayers" / "exercise" / "my_skyscraper.usda"
if HAVE_COMP_ARCS and _my_sky.exists():
    root = Sdf.Layer.FindOrOpen(str(_my_sky))
    walk_sublayers(root)
else:
    print("Skipping: my_skyscraper.usda was not authored (sublayers exercise assets missing).")

/Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/sublayers/exercise/my_skyscraper.usda
  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/sublayers/exercise/contents/shading.usd
  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/sublayers/exercise/contents/geometry.usd


## 5.2 References and payloads

Sublayers *include* a whole layer verbatim, which means they cannot bring the same content in twice at different namespace locations. When you need multiple independent copies of an asset — three chairs, twenty cars, an entire city of repeated buildings — you reach for **references** or **payloads**.

- A **reference** grafts another layer's prim hierarchy onto a target prim, rewriting paths so each graft lives under its own namespace.
- A **payload** is a reference that can be *unloaded*. You pay the cost of composing the referenced content only when you ask for it.

References are always composed on the stage. Payloads are weaker than references in LIVRPS, so if you need a specific strength ordering you sometimes pick one over the other for that reason alone.

### 5.2.1 A simple reference scene

`references/simple_example/references_simple.usd` references `red_cube.usd` into three different prims (external reference) and then internally references a `blue_cube_01` prim into two more (internal reference). Walk the composition with `UsdPrimCompositionQuery`:

In [8]:
from pxr import Usd

if require_assets("references/simple_example/references_simple.usd"):
    stage = Usd.Stage.Open(str(ASSET_ROOT / "references" / "simple_example" / "references_simple.usd"))
    print("Top-level prims under /World:")
    world = stage.GetPrimAtPath("/World")
    for child in world.GetChildren():
        print("  ", child.GetPath(), "type=", child.GetTypeName())

    print("\nComposition arcs feeding /World/red_cube_01:")
    target = stage.GetPrimAtPath("/World/red_cube_01")
    query = Usd.PrimCompositionQuery(target)
    for arc in query.GetCompositionArcs():
        print("  arc type =", arc.GetArcType(),
              "| target =", arc.GetTargetPrimPath(),
              "| layer  =", arc.GetTargetLayer().identifier if arc.GetTargetLayer() else None)

Top-level prims under /World:
   /World/red_cube_01 type= Xform
   /World/red_cube_02 type= Xform
   /World/red_cube_03 type= Xform
   /World/blue_cube_01 type= Xform
   /World/blue_cube_02 type= Xform
   /World/blue_cube_03 type= Xform

Composition arcs feeding /World/red_cube_01:
  arc type = Pcp.ArcTypeRoot | target = /World/red_cube_01 | layer  = /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/references/simple_example/references_simple.usd
  arc type = Pcp.ArcTypeReference | target = /World | layer  = /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/references/simple_example/red_cube.usd


**What just happened:** `UsdPrimCompositionQuery` is the programmatic equivalent of usdview's *Composition* tab. It lists every arc contributing to a prim and where it comes from — invaluable when a scene is misbehaving and you need to find the offending layer.

### 5.2.2 Building a row of skyscrapers from references

The referenced-city exercise builds a small city block by referencing two skyscraper assets multiple times with different translations. Each reference is a graft — edits to the target prim do not propagate back to the source asset, and a translate on the reference only affects that single instance.

In [9]:
import os
from pathlib import Path
from pxr import Usd, UsdGeom, Gf

if require_assets("lib/assets/envir/city"):
    asset_library = ASSET_ROOT / "lib" / "assets"
    working_dir = ASSET_ROOT / "references" / "exercise"
    working_dir.mkdir(parents=True, exist_ok=True)
    out_path = working_dir / "city.usda"
    if out_path.exists():
        out_path.unlink()

    def relpath(p: Path) -> str:
        # Forward slashes for cross-platform USD asset paths.
        return Path(os.path.relpath(p, working_dir)).as_posix()

    skyscraperA = asset_library / "envir" / "city" / "skyscraperA" / "skyscraperA.usd"
    skyscraperE = asset_library / "envir" / "city" / "skyscraperE" / "skyscraperE.usd"

    stage = Usd.Stage.CreateNew(str(out_path))
    UsdGeom.SetStageMetersPerUnit(stage, UsdGeom.LinearUnits.centimeters)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    UsdGeom.Xform.Define(stage, "/World")

    if skyscraperA.exists():
        sA_rel = relpath(skyscraperA)
        p = UsdGeom.Xform.Define(stage, "/World/skyscraperA_01")
        p.GetPrim().GetReferences().AddReference(sA_rel)
        p = UsdGeom.Xform.Define(stage, "/World/skyscraperA_02")
        p.GetPrim().GetReferences().AddReference(sA_rel)
        UsdGeom.XformCommonAPI(p).SetTranslate(Gf.Vec3d(180, 0, 0))

    if skyscraperE.exists():
        sE_rel = relpath(skyscraperE)
        p = UsdGeom.Xform.Define(stage, "/World/skyscraperE_01")
        p.GetPrim().GetReferences().AddReference(sE_rel)
        UsdGeom.XformCommonAPI(p).SetTranslate(Gf.Vec3d(340, 0, 0))

    stage.Save()
    print("Wrote", out_path)
    print("\nComposed city block:")
    for prim in stage.GetPrimAtPath("/World").GetChildren():
        refs = prim.GetMetadata("references")
        print("  ", prim.GetPath(), "references=", refs)

Wrote /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/references/exercise/city.usda

Composed city block:
   /World/skyscraperA_01 references= SdfReferenceListOp(Prepended Items: [SdfReference(../../lib/assets/envir/city/skyscraperA/skyscraperA.usd, , SdfLayerOffset(0, 1), {})])
   /World/skyscraperA_02 references= SdfReferenceListOp(Prepended Items: [SdfReference(../../lib/assets/envir/city/skyscraperA/skyscraperA.usd, , SdfLayerOffset(0, 1), {})])
   /World/skyscraperE_01 references= SdfReferenceListOp(Prepended Items: [SdfReference(../../lib/assets/envir/city/skyscraperE/skyscraperE.usd, , SdfLayerOffset(0, 1), {})])


**What just happened:** one script authored a scene that would be three completely duplicated asset copies on disk if it weren't for references. Adjusting `skyscraperA.usd` (fix its roof, retune its materials) now updates every scene that references it — the modularity that made USD worth building.

### 5.2.3 References FAQ: sublayers vs references, and `prepend`

- **Why not just sublayer the asset twice?** Sublayers don't re-path anything — the two copies would land at the same prim paths and clobber each other. References re-path the graft to whatever prim you attach them to, so you get real siblings.
- **Why is the authored arc `prepend references` instead of `add references`?** `prepend` puts this reference at the strong end of the list when composition stacks arcs, which is the intuitive "my opinion wins" behavior. `UsdReferences.AddReference()` defaults to prepend for exactly this reason. Other positions exist (`append`, explicit indices) but are rarely needed.

### 5.2.4 Payloads: deferred, unloadable references

Payloads use the same grafting mechanics as references, but the composition engine only pulls the target layer's contents onto the stage when the payload is **loaded**. Unloading a payload removes the composed descendants from the stage until you load it again. This is how you open a 10,000-prim city scene in a second and only pay for the block you are inspecting.

In [10]:
import shutil
from pathlib import Path
from pxr import Usd, UsdGeom, Gf

if require_assets("lib/assets/envir/city/sm_bldgF/sm_bldgF.usd"):
    working_dir = ASSET_ROOT / "payloads" / "exercise"
    working_dir.mkdir(parents=True, exist_ok=True)

    # Stage a sibling copy of sm_bldgF.usd so the payload path resolves.
    bldg_src_dir = ASSET_ROOT / "lib" / "assets" / "envir" / "city" / "sm_bldgF"
    bldg_dst_dir = working_dir / "sm_bldgF"
    if not bldg_dst_dir.exists():
        shutil.copytree(bldg_src_dir, bldg_dst_dir)
    alias = working_dir / "sm_bldgF.usd"
    if not alias.exists():
        alias.write_bytes((bldg_dst_dir / "sm_bldgF.usd").read_bytes())

    out_path = working_dir / "city.usda"
    if out_path.exists():
        out_path.unlink()

    stage = Usd.Stage.CreateNew(str(out_path))
    UsdGeom.SetStageMetersPerUnit(stage, UsdGeom.LinearUnits.centimeters)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
    UsdGeom.Xform.Define(stage, "/World")

    for i, tx in enumerate([0, 180, 340], start=1):
        prim = UsdGeom.Xform.Define(stage, f"/World/sm_bldgF_{i:02d}")
        prim.GetPrim().GetPayloads().AddPayload("./sm_bldgF.usd")
        UsdGeom.XformCommonAPI(prim).SetTranslate(Gf.Vec3d(tx, 0, 0))
    stage.Save()

    # Open the composed city and see what it looks like fully loaded vs fully unloaded.
    loaded_stage = Usd.Stage.Open(str(out_path))
    print("Loaded prim count  :", sum(1 for _ in loaded_stage.Traverse()))

    loaded_stage.GetPseudoRoot().Unload()
    print("Unloaded prim count:", sum(1 for _ in loaded_stage.Traverse()))

    # Load just one payload back.
    one = loaded_stage.GetPrimAtPath("/World/sm_bldgF_01")
    one.Load()
    print("Loaded-one count   :", sum(1 for _ in loaded_stage.Traverse()))

Loaded prim count  : 49
Unloaded prim count: 1
Loaded-one count   : 17


**What just happened:** the same graft-a-building-three-times idea, but authored with `GetPayloads().AddPayload()` instead of `GetReferences().AddReference()`. Because they are payloads we can `Unload()` them en masse via the pseudo-root and then selectively `Load()` only the buildings we care about — the basis of deferred loading on gigantic scenes.

## 5.3 Encapsulation

When you reference an asset, the composition engine copies the sub-hierarchy under the reference source prim and re-paths it into the destination namespace. Anything *outside* that sub-hierarchy — sibling prims, class prims, loose materials parked at `/Looks/...` above the asset root — is **not** brought along. A prim is said to be **encapsulated** when every prim it depends on (materials, shaders, class sources) lives *beneath* the referenced target.

Good rule of thumb: **put everything your asset needs under its default prim.** If a material binding or inherit target is outside the default prim, references into another layer stack will break and you will see a composition warning about "refers to a path outside the scope of the reference."

![encapsulation diagram](../docs/images/composition-arcs/image102.png)

In [11]:
from pxr import Usd

if require_assets("references/encapsulation_example/encapsulated_GOOD.usd",
                  "references/encapsulation_example/unencapsulated_BAD.usd",
                  "references/encapsulation_example/references_encapsulation.usd"):
    enc_dir = ASSET_ROOT / "references" / "encapsulation_example"
    good_stage = Usd.Stage.Open(str(enc_dir / "encapsulated_GOOD.usd"))
    bad_stage  = Usd.Stage.Open(str(enc_dir / "unencapsulated_BAD.usd"))

    def top_level(stage, label):
        print(f"{label}: default prim = {stage.GetDefaultPrim().GetPath() if stage.GetDefaultPrim() else None}")
        for child in stage.GetPseudoRoot().GetChildren():
            print("   /" + child.GetName(), "type=", child.GetTypeName() or "<over>")

    top_level(good_stage, "encapsulated_GOOD.usd")
    print()
    top_level(bad_stage,  "unencapsulated_BAD.usd")

    # Open the layer that references both. The bad one logs a composition warning
    # about material bindings escaping the reference scope.
    ref_stage = Usd.Stage.Open(str(enc_dir / "references_encapsulation.usd"))
    print("\nComposed references_encapsulation.usd prims:")
    for prim in ref_stage.Traverse():
        print("  ", prim.GetPath(), prim.GetTypeName())

encapsulated_GOOD.usd: default prim = /World
   /World type= Xform

unencapsulated_BAD.usd: default prim = /World
   /World type= Xform
   /Looks type= Scope

Composed references_encapsulation.usd prims:
   /World Xform
   /World/encapsulated_GOOD Xform
   /World/encapsulated_GOOD/Cube Cube
   /World/encapsulated_GOOD/RedMaterial Material
   /World/encapsulated_GOOD/RedMaterial/Shader Shader
   /World/unencapsulated_BAD_02 Cube
   /World/unencapsulated_BAD_01 Xform
   /World/unencapsulated_BAD_01/Cube Cube


**What just happened:** `encapsulated_GOOD.usd` keeps its `Looks/RedMaterial` *under* `/World` and references cleanly. `unencapsulated_BAD.usd` parks the material at `/Looks/RedMaterial` — a sibling of `/World` — so when someone references `/World` into another scene, the material binding dangles and USD emits a warning. Same data on disk; different namespace layouts; very different referencing behavior.

## 5.4 Variant sets

A **variant set** is a named switch on a prim. Each **variant** inside it is a bundle of opinions — overrides, new descendant prims, even new composition arcs — that only composes onto the stage when that variant is selected. Variant sets are how a single character asset ships with "hat / no hat" geometry, "dry / wet" materials, and every combination in between.

Key constraints:

- A variant set lives *on a prim* and its variants can affect that prim plus any descendants.
- Variants can: sparsely override existing properties, define new descendant prims, or author new composition arcs (e.g. reference a different material from each variant).
- **Lightweight switches** (attribute values, visibility) only re-resolve values — fast. **Heavyweight switches** (swapping references, toggling activation) rebuild the composition index — slower. Pick the cheaper mechanism when you expect frequent toggling.

### 5.4.1 Reading an existing variant set

`variant_sets/simple_example/variant_sets_simple.usd` has a single `color` variant set on `/World` with two variants (`red`, `blue`) that each override the shader's `inputs:diffuseColor`. You can inspect and toggle it purely in Python:

In [12]:
from pxr import Usd

if require_assets("variant_sets/simple_example/variant_sets_simple.usd"):
    stage = Usd.Stage.Open(str(ASSET_ROOT / "variant_sets" / "simple_example" / "variant_sets_simple.usd"))
    world = stage.GetDefaultPrim() or stage.GetPrimAtPath("/World")
    vsets = world.GetVariantSets()
    print("Variant sets on", world.GetPath(), ":", vsets.GetNames())

    vset = vsets.GetVariantSet(vsets.GetNames()[0])
    print("Variants available    :", vset.GetVariantNames())
    print("Current selection     :", vset.GetVariantSelection())

    shader = next((p for p in stage.Traverse() if p.GetTypeName() == "Shader"), None)
    if shader:
        diffuse = shader.GetAttribute("inputs:diffuseColor")
        for name in vset.GetVariantNames():
            vset.SetVariantSelection(name)
            print(f"  {name:>5} -> diffuseColor = {diffuse.Get()}")

Variant sets on /World : ['color']
Variants available    : ['blue', 'green', 'red']
Current selection     : 
   blue -> diffuseColor = (0, 0, 1)
  green -> diffuseColor = (0, 1, 0)
    red -> diffuseColor = (1, 0, 0)


### 5.4.2 Authoring a variant set from scratch

This is the streetlamp exercise. We create a `lights` variant set on the asset's default prim, add `on` and `off` variants, and inside each variant's edit context we either pump the light intensity up or zero it out.

In [13]:
import shutil
from pathlib import Path
from pxr import Usd, UsdLux, UsdShade, Sdf

if require_assets("variant_sets/exercise/street_lamp_dbl.usd"):
    working_dir = ASSET_ROOT / "variant_sets" / "exercise"
    lamp_src = working_dir / "street_lamp_dbl.usd"
    # Work on a copy so reruns do not stomp the pristine exercise file.
    lamp_work = working_dir / "_nb_street_lamp_dbl.usda"
    shutil.copyfile(lamp_src, lamp_work)

    stage = Usd.Stage.Open(str(lamp_work))
    root = stage.GetDefaultPrim()

    def toggle_lights(on: bool):
        intensity = 100.0 if on else 0.0
        emissive  = (1.0, 1.0, 1.0) if on else (0.0, 0.0, 0.0)
        lights_scope = stage.GetPrimAtPath(root.GetPath().AppendPath("Lights"))
        if lights_scope and lights_scope.IsValid():
            for prim in Usd.PrimRange(lights_scope):
                if prim.HasAPI(UsdLux.LightAPI):
                    UsdLux.LightAPI(prim).GetIntensityAttr().Set(intensity)
        shader_prim = stage.GetPrimAtPath(root.GetPath().AppendPath("Looks/light/light"))
        if shader_prim and shader_prim.IsValid():
            shader = UsdShade.Shader(shader_prim)
            shader.CreateInput("emissiveColor", Sdf.ValueTypeNames.Color3f).Set(emissive)

    vset = root.GetVariantSets().AddVariantSet("lights")
    vset.AddVariant("on")
    vset.AddVariant("off")

    vset.SetVariantSelection("on")
    with vset.GetVariantEditContext():
        toggle_lights(True)

    vset.SetVariantSelection("off")
    with vset.GetVariantEditContext():
        toggle_lights(False)

    # Pick a baseline selection so downstream scenes inherit something sensible.
    vset.SetVariantSelection("off")
    out_path = working_dir / "street_lamp_dbl_vset.usda"
    stage.GetRootLayer().Export(str(out_path))
    print("Wrote", out_path)
    print("Variants on root prim:", root.GetVariantSets().GetNames(),
          "| selection =", vset.GetVariantSelection())

Wrote /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/composition_arcs/variant_sets/exercise/street_lamp_dbl_vset.usda
Variants on root prim: ['lights'] | selection = off


**What just happened:** `GetVariantEditContext()` is the critical idiom. Any edits you make inside the `with` block are routed into the currently selected variant's spec instead of the root layer — that's how per-variant opinions get authored. Outside the block, edits land on whichever layer the stage's EditTarget points at (the root layer by default).

## 5.5 Inherits and specializes

Inherits and specializes are the two "broadcast" arcs. Both let you author an opinion on a single source prim and have it apply to every prim that points at that source — without you editing each destination individually. They differ in *when* they win:

- **Inherit** arcs are *strong*. The source prim's opinion wins over the destination prim's own authored value (in the same context). Think "push this value onto every instance."
- **Specializes** arcs are *weak*. The source prim's opinion serves as a **fallback**; any authored opinion on the destination prim overrides it. Think "give every instance this default, unless it has been customized."

Both typically target a `class` prim (authored with the `class` specifier) so it doesn't render by default and signals "I exist to be inherited/specialized from."

### 5.5.1 Simple inherits: broadcast a color to every pyramid cube

In `inherits/simple_example/`, `cube.usd` defines a class `/_cube_asset` that the `World` prim inherits from, and `cube_pyramid.usd` references `cube.usd` three times. Authoring `inputs:diffuseColor` on `/_cube_asset` in `inherits_simple.usd` turns *all three* pyramid cubes yellow at once — without editing any of them directly — while `unaffected_scenario.usd` (which references `cube_pyramid.usd` from a different layer stack) stays unchanged.

In [14]:
from pxr import Usd, UsdShade

def print_pyramid_colors(stage_path):
    stage = Usd.Stage.Open(str(stage_path))
    shaders = [p for p in stage.Traverse() if p.GetTypeName() == "Shader"]
    print(f"-- {stage_path.name} -- ({len(shaders)} shaders)")
    for s in shaders:
        diffuse = s.GetAttribute("inputs:diffuseColor")
        if diffuse and diffuse.HasAuthoredValue():
            print("  ", s.GetPath(), "->", diffuse.Get())

if require_assets("inherits/simple_example/inherits_simple.usd",
                  "inherits/simple_example/unaffected_scenario.usd"):
    simple_dir = ASSET_ROOT / "inherits" / "simple_example"
    print_pyramid_colors(simple_dir / "inherits_simple.usd")
    print()
    print_pyramid_colors(simple_dir / "unaffected_scenario.usd")

-- inherits_simple.usd -- (3 shaders)
   /World/cube_pyramid/cube_01/CubeMaterial/Shader -> (1, 1, 0)
   /World/cube_pyramid/cube_02/CubeMaterial/Shader -> (1, 1, 0)
   /World/cube_pyramid/cube_03/CubeMaterial/Shader -> (1, 1, 0)

-- unaffected_scenario.usd -- (3 shaders)
   /World/cube_pyramid/cube_01/CubeMaterial/Shader -> (0, 0, 1)
   /World/cube_pyramid/cube_02/CubeMaterial/Shader -> (0, 1, 0)
   /World/cube_pyramid/cube_03/CubeMaterial/Shader -> (1, 0, 0)


### 5.5.2 Authoring an inherits arc: streetlamp recolor

The practical exercise gives a streetlamp asset an inherits arc to the class `/_street_lamp_dbl`, then a downstream scenario layer overrides that class to tint every streetlamp light yellow — but only in that one scenario. Other scenarios that reference the same asset are untouched.

In [15]:
import shutil
from pathlib import Path
from pxr import Usd, UsdLux, UsdShade, Sdf

if require_assets("inherits/exercise/street_lamp_dbl.usd",
                  "inherits/exercise/scenario_02.usd"):
    working_dir = ASSET_ROOT / "inherits" / "exercise"

    # Work on copies so repeat notebook runs stay idempotent.
    lamp_work = working_dir / "_nb_street_lamp_dbl.usda"
    scn2_work = working_dir / "_nb_scenario_02.usda"
    shutil.copyfile(working_dir / "street_lamp_dbl.usd", lamp_work)
    shutil.copyfile(working_dir / "scenario_02.usd",     scn2_work)

    # Step 1: add the inherits arc on the asset.
    asset_stage = Usd.Stage.Open(str(lamp_work))
    class_prim = asset_stage.CreateClassPrim("/_street_lamp_dbl")
    asset_stage.GetDefaultPrim().GetInherits().AddInherit(class_prim.GetPath())
    asset_stage.GetRootLayer().Save()
    print("Asset inherits authored:",
          asset_stage.GetDefaultPrim().GetInherits().GetAllDirectInherits())

    # Step 2: override the class in a downstream scenario layer.
    scenario2 = Usd.Stage.Open(str(scn2_work))
    class_over = scenario2.OverridePrim("/_street_lamp_dbl")

    for light_name in ("sphere_light_01", "sphere_light_02"):
        lp = scenario2.OverridePrim(class_over.GetPath().AppendPath(f"Lights/{light_name}"))
        UsdLux.LightAPI(lp).CreateColorAttr((0.5, 0.4, 0.1))

    shader_over = scenario2.OverridePrim(class_over.GetPath().AppendPath("Looks/light/light"))
    UsdShade.Shader(shader_over).CreateInput("emissiveColor", Sdf.ValueTypeNames.Color3f).Set((0.5, 0.4, 0.1))
    scenario2.GetRootLayer().Save()

    print("scenario_02 overrides now target:", class_over.GetPath(),
          "children=", [c.GetName() for c in class_over.GetChildren()])

Asset inherits authored: [Sdf.Path('/_street_lamp_dbl')]
scenario_02 overrides now target: /_street_lamp_dbl children= []


**What just happened:** we tagged the streetlamp asset as "inherits from `/_street_lamp_dbl`" and then, in a completely different scenario layer, overrode that class prim. The override broadcasts to every streetlamp in scenario 2 without editing any individual lamp — and scenario 1, which references the same lamp, never sees the change.

### 5.5.3 Specializes: broadcast a *fallback*

Specializes arcs look structurally identical (`prim.GetSpecializes().AddSpecialize(class_path)`) but they land at the *weak* end of LIVRPS. If a destination prim has its own authored opinion for an attribute, the specializes arc loses; it only contributes when no stronger opinion exists.

The roads exercise uses this to give every road piece a default `osm:street:maxspeed = 30`, while still allowing individual side streets to be overridden to `20` without the class clobbering them.

In [16]:
from pxr import Usd, UsdGeom, Sdf

# This cell is a self-contained miniature of the specializes exercise so it works
# even if the shipped roads assets aren't available.
sketch_path = NB_DIR / "_assets" / "05_specializes_sketch.usda"
if sketch_path.exists():
    sketch_path.unlink()
stage = Usd.Stage.CreateNew(str(sketch_path))

# 1) A class prim that holds the fallback maxspeed.
class_prim = stage.CreateClassPrim("/_osm_street_data")
attr = class_prim.CreateAttribute("osm:street:maxspeed", Sdf.ValueTypeNames.Int, custom=True)
attr.Set(30)

# 2) A small set of road prims that specialize from the class.
UsdGeom.Xform.Define(stage, "/World")
for name, is_side in [("road_main", False), ("road_sideA", True), ("road_sideB", True)]:
    road = UsdGeom.Mesh.Define(stage, f"/World/{name}")
    road.GetPrim().GetSpecializes().AddSpecialize(class_prim.GetPath())
    if is_side:
        # Local opinion — stronger than the specializes fallback.
        road.GetPrim().GetAttribute("osm:street:maxspeed").Set(20)

stage.GetRootLayer().Save()

for prim in stage.GetPrimAtPath("/World").GetChildren():
    print(prim.GetPath(), "maxspeed =", prim.GetAttribute("osm:street:maxspeed").Get())

# Now simulate a director note: raise the global fallback to 40.
class_prim.GetAttribute("osm:street:maxspeed").Set(40)
print("\nAfter bumping class fallback to 40:")
for prim in stage.GetPrimAtPath("/World").GetChildren():
    print(prim.GetPath(), "maxspeed =", prim.GetAttribute("osm:street:maxspeed").Get())

/World/road_main maxspeed = 30
/World/road_sideA maxspeed = 20
/World/road_sideB maxspeed = 20

After bumping class fallback to 40:
/World/road_main maxspeed = 40
/World/road_sideA maxspeed = 20
/World/road_sideB maxspeed = 20


**What just happened:** the main road has no authored opinion, so it tracks the class fallback (30, then 40). The two side roads have their own local opinions of 20, so the class bump to 40 leaves them alone. That is specializes in one sentence: *fallback values that refined destinations can ignore.*

### 5.5.4 Encapsulation and refinement

Inherits and specializes can target either encapsulated (under the asset's default prim) or unencapsulated (outside it) source prims. That choice determines the *scope* of the broadcast when the asset is referenced into another layer stack:

- **Global refinement** — inherit target is *unencapsulated*. Its path never gets re-pathed on reference, so overriding it in any downstream layer broadcasts to every instance anywhere in that layer stack. Powerful but wide-reaching.
- **Local refinement** — inherit target is *encapsulated* (lives inside the default prim). Its path *is* re-pathed on every reference, so each reference ends up with its own private class. Overrides only affect the single reference you target.

The shipped `inherits/refinement_example/` folder has `global_refinement.usd` and `local_refinement.usd` side by side — same cube pyramid structure, different target path layouts, dramatically different broadcast scopes.

In [17]:
from pxr import Usd, Sdf

def list_class_prims(stage_path):
    stage = Usd.Stage.Open(str(stage_path))
    print(f"-- {stage_path.name} --")
    for prim in stage.TraverseAll():
        if prim.GetSpecifier() == Sdf.SpecifierClass:
            print("  class:", prim.GetPath())

if require_assets("inherits/refinement_example/global_refinement.usd",
                  "inherits/refinement_example/local_refinement.usd"):
    ref_dir = ASSET_ROOT / "inherits" / "refinement_example"
    list_class_prims(ref_dir / "global_refinement.usd")
    print()
    list_class_prims(ref_dir / "local_refinement.usd")


-- global_refinement.usd --
  class: /World/cube_pyramid_01/_local_cube
  class: /World/cube_pyramid_02/_local_cube
  class: /World/cube_pyramid_03/_local_cube

-- local_refinement.usd --
  class: /World/cube_pyramid_01/_local_cube
  class: /World/cube_pyramid_02/_local_cube
  class: /World/cube_pyramid_03/_local_cube


**What just happened:** `TraverseAll()` (unlike `Traverse()`) visits class prims too, which is how you inspect abstract hierarchies that never image. Notice how the global example keeps its class prim at the root (`/_cube_asset`), while the local example parks its class inside a specific pyramid instance — which is exactly what re-paths on reference.

## 5.6 LIVRPS strength ordering

LIVRPS (pronounced "liver peas") is the acronym that captures every composition operation in the order it is applied when building a prim's composed value. From strongest to weakest:

1. **L**ocal — opinions authored directly in the root layer or any of its (recursive) sublayers. The layer stack.
2. **I**nherits — opinions from inherit arcs.
3. **V**ariant sets — opinions from the selected variant.
4. **R**eferences — opinions from reference arcs.
5. **P**ayloads — opinions from payload arcs (always weaker than references).
6. **S**pecializes — opinions from specialize arcs (the weakest).

LIVRPS applies *recursively* inside each layer stack. And "local" includes every sublayer of every sublayer, walked depth-first — they are all considered direct opinions on the same layer stack.

In [18]:
from pxr import Usd, Sdf, UsdGeom
from pathlib import Path

# Build a tiny scene that exercises three LIVRPS levels and then trace which
# layer each opinion comes from via UsdAttribute.GetPropertyStack().
work = NB_DIR / "_assets"
class_layer_path = work / "05_livrps_class.usda"
ref_layer_path   = work / "05_livrps_asset.usda"
root_layer_path  = work / "05_livrps_root.usda"
for p in (class_layer_path, ref_layer_path, root_layer_path):
    if p.exists():
        p.unlink()

# Weakest: a referenced asset layer that sets radius = 1.0.
asset_stage = Usd.Stage.CreateNew(str(ref_layer_path))
UsdGeom.Sphere.Define(asset_stage, "/Asset").GetRadiusAttr().Set(1.0)
asset_stage.SetDefaultPrim(asset_stage.GetPrimAtPath("/Asset"))
asset_stage.GetRootLayer().Save()

# Middle: a class prim layer that sets radius = 2.0 via inherits.
# (Really the class is authored in the root layer below, since inherits are
# resolved per-layer-stack. Kept here as a plain sibling for inspection.)

# Strongest: the root layer authors radius = 3.0 directly on the prim.
root_stage = Usd.Stage.CreateNew(str(root_layer_path))
root_prim = UsdGeom.Sphere.Define(root_stage, "/World/Ball")
root_prim.GetPrim().GetReferences().AddReference("./05_livrps_asset.usda")

cls = root_stage.CreateClassPrim("/_BallClass")
UsdGeom.Sphere(cls).CreateRadiusAttr(2.0)
root_prim.GetPrim().GetInherits().AddInherit(cls.GetPath())

# Local opinion — strongest of all in LIVRPS.
root_prim.GetRadiusAttr().Set(3.0)
root_stage.GetRootLayer().Save()

# Reopen and inspect.
stage = Usd.Stage.Open(str(root_layer_path))
ball = stage.GetPrimAtPath("/World/Ball")
radius = ball.GetAttribute("radius")
print("Composed radius:", radius.Get())
print("\nProperty stack (strongest -> weakest):")
for spec in radius.GetPropertyStack(Usd.TimeCode.Default()):
    print(f"  {spec.layer.identifier:60s} default={spec.default}")

Composed radius: 3.0

Property stack (strongest -> weakest):
  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/05_livrps_root.usda default=3.0
  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/05_livrps_root.usda default=2.0
  /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/05_creating_composition_arcs/_assets/05_livrps_asset.usda default=1.0


**What just happened:** we stacked three opinions for the same `radius` — a local value (3.0) in the root layer, an inherited class value (2.0), and a referenced asset value (1.0). LIVRPS orders them local → inherits → references, so 3.0 wins. Comment out the local opinion and 2.0 wins; remove the class too and 1.0 wins. `GetPropertyStack()` shows the full chain every time.

Tracing through LIVRPS is the composition debugging skill. Every mystery in a USD scene — "why is my cube green when I set it red?" — boils down to asking *which arc produced the winning opinion and is there a stronger arc I missed?*

## Key takeaways

- **Sublayers** include layers verbatim — great for parallel workstreams, useless for repeated assets.
- **References** graft a hierarchy with path translation, so the same asset can appear many times in the same scene.
- **Payloads** are references that can be loaded/unloaded on demand; use them for deferred loading of heavy assets.
- **Encapsulation** (everything under the default prim) is what makes references survive the layer-stack boundary without dangling bindings.
- **Variant sets** bundle opinions behind a named switch; use `GetVariantEditContext()` to author into a variant. Prefer lightweight switches (attribute/visibility) over heavyweight ones (swapping references, activation) when toggling is frequent.
- **Inherits** broadcast *strong* overrides to every prim that inherits from a source class; **specializes** broadcast *weak* fallbacks that authored opinions can shadow.
- **LIVRPS** (Local, Inherits, Variants, References, Payloads, Specializes) is the fixed strength order. `UsdAttribute.GetPropertyStack()` and `UsdPrimCompositionQuery` are the two essential debugging tools.
- `_assets/composition_arcs/` now contains the full exercise content tree — feel free to open those files in usdview to cross-check anything you saw programmatically here.

## Next up → `06_asset_structure.ipynb`

Having the arcs is half the battle; the other half is knowing *how to arrange them inside an asset* so references, payloads, and variant sets all cooperate. Notebook 06 walks through the canonical asset structure pattern used throughout the course's lib assets.